In [4]:
# STEP 1. 개발 환경 점검
# 읽기 전용 점검: API 호출, 폴더 생성, 파일 수정 없음

import sys
import os
import platform
import importlib
from importlib.metadata import version, PackageNotFoundError
from pathlib import Path


# --------------------------------------------------
# 1) 프로젝트 루트 탐색
# --------------------------------------------------
current_path = Path.cwd().resolve()

# 현재 위치부터 상위 폴더를 확인하여 .env가 있는 곳을 프로젝트 루트로 판단
project_root = next(
    (path for path in [current_path, *current_path.parents]
     if (path / ".env").is_file()),
    current_path
)

env_path = project_root / ".env"
gitignore_path = project_root / ".gitignore"

raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
logs_dir = project_root / "logs"

backup_json_path = (
    raw_dir / "naver_trend_raw_20261119_215045.json"
)


# --------------------------------------------------
# 2) 필수 패키지 점검
# --------------------------------------------------
required_packages = {
    "requests": "requests",
    "python-dotenv": "dotenv",
    "pandas": "pandas",
    "ipykernel": "ipykernel",
}

package_results = {}

for package_name, import_name in required_packages.items():
    try:
        importlib.import_module(import_name)

        try:
            installed_version = version(package_name)
        except PackageNotFoundError:
            installed_version = "버전 확인 불가"

        package_results[package_name] = {
            "installed": True,
            "version": installed_version,
            "error": None,
        }

    except Exception as error:
        package_results[package_name] = {
            "installed": False,
            "version": None,
            "error": f"{type(error).__name__}: {error}",
        }


# --------------------------------------------------
# 3) .env 로딩 및 인증정보 존재 여부 점검
# --------------------------------------------------
env_loaded = False

if env_path.is_file() and package_results["python-dotenv"]["installed"]:
    from dotenv import load_dotenv

    # 현재 프로세스에 이미 설정된 환경변수는 덮어쓰지 않음
    env_loaded = load_dotenv(
        dotenv_path=env_path,
        override=False,
    )

# 프로젝트에서 사용할 수 있는 일반적인 환경변수 이름 후보
client_id_names = (
    "NAVER_CLIENT_ID",
    "X_NCP_APIGW_API_KEY_ID",
    "NCP_CLIENT_ID",
)

client_secret_names = (
    "NAVER_CLIENT_SECRET",
    "X_NCP_APIGW_API_KEY",
    "NCP_CLIENT_SECRET",
)

detected_client_id_name = next(
    (name for name in client_id_names if os.getenv(name)),
    None,
)

detected_client_secret_name = next(
    (name for name in client_secret_names if os.getenv(name)),
    None,
)

credentials_ready = (
    detected_client_id_name is not None
    and detected_client_secret_name is not None
)


# --------------------------------------------------
# 4) Jupyter 커널과 가상환경 점검
# --------------------------------------------------
python_executable = Path(sys.executable).resolve()

uses_dot_venv = any(
    part.lower() == ".venv"
    for part in python_executable.parts
)

python_version = platform.python_version()
python_version_ok = python_version == "3.12.10"


# --------------------------------------------------
# 5) .gitignore 보안 설정 점검
# --------------------------------------------------
gitignore_patterns = []

if gitignore_path.is_file():
    gitignore_patterns = [
        line.strip().replace("\\", "/")
        for line in gitignore_path.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
        if line.strip() and not line.strip().startswith("#")
    ]

env_ignore_ok = any(
    pattern in {".env", "/.env", ".env*", "/.env*"}
    for pattern in gitignore_patterns
)

venv_ignore_ok = any(
    pattern.rstrip("/") in {".venv", "/.venv"}
    for pattern in gitignore_patterns
)


# --------------------------------------------------
# 6) 전체 판정 결과
# --------------------------------------------------
checks = [
    ("Python 버전 3.12.10", python_version_ok),
    ("Jupyter 커널이 .venv 사용", uses_dot_venv),
    (
        "필수 패키지 import",
        all(result["installed"] for result in package_results.values()),
    ),
    (".env 파일 존재", env_path.is_file()),
    ("인증정보 2종 로딩", credentials_ready),
    ("기존 원본 JSON 존재", backup_json_path.is_file()),
    ("data/raw 폴더 존재", raw_dir.is_dir()),
    ("data/processed 폴더 존재", processed_dir.is_dir()),
    ("logs 폴더 존재", logs_dir.is_dir()),
    (".env Git 제외 설정", env_ignore_ok),
    (".venv Git 제외 설정", venv_ignore_ok),
]


# --------------------------------------------------
# 7) 결과 출력
# --------------------------------------------------
print("=" * 70)
print("STEP 1 개발 환경 점검 결과")
print("=" * 70)

print(f"프로젝트 루트      : {project_root}")
print(f"Python 버전        : {python_version}")
print(f"Python 실행 파일   : {python_executable}")
print(f"운영체제           : {platform.platform()}")

print("\n[필수 패키지]")
for package_name, result in package_results.items():
    if result["installed"]:
        print(f"[정상] {package_name}: {result['version']}")
    else:
        print(f"[확인 필요] {package_name}: {result['error']}")

print("\n[환경변수]")
print(f".env 파일          : {'존재' if env_path.is_file() else '없음'}")
print(f".env 로딩 시도     : {'완료' if env_path.is_file() else '미실행'}")
print(
    "Client ID 변수    : "
    + (detected_client_id_name if detected_client_id_name else "확인되지 않음")
)
print(
    "Client Secret 변수: "
    + (
        detected_client_secret_name
        if detected_client_secret_name
        else "확인되지 않음"
    )
)
print("※ 인증키 값 자체는 보안을 위해 출력하지 않습니다.")

print("\n[파일 및 폴더]")
print(f"기존 원본 JSON     : {backup_json_path}")
print(
    f"JSON 존재 여부     : "
    f"{'존재' if backup_json_path.is_file() else '없음'}"
)
print(f"data/raw           : {'존재' if raw_dir.is_dir() else '없음'}")
print(
    f"data/processed     : "
    f"{'존재' if processed_dir.is_dir() else '없음'}"
)
print(f"logs               : {'존재' if logs_dir.is_dir() else '없음'}")

print("\n[최종 점검표]")
for check_name, passed in checks:
    status = "정상" if passed else "확인 필요"
    print(f"[{status}] {check_name}")

failed_checks = [
    check_name
    for check_name, passed in checks
    if not passed
]

print("\n" + "=" * 70)

if failed_checks:
    print(f"확인 필요한 항목: {len(failed_checks)}개")
    for item in failed_checks:
        print(f"- {item}")
else:
    print("STEP 1의 모든 점검 항목이 정상입니다.")

print("=" * 70)

STEP 1 개발 환경 점검 결과
프로젝트 루트      : C:\Users\user\Desktop\KDT_2026\프로젝트\03_융합_프로젝트\01_DB
Python 버전        : 3.12.10
Python 실행 파일   : C:\Users\user\Desktop\KDT_2026\프로젝트\03_융합_프로젝트\01_DB\.venv\Scripts\python.exe
운영체제           : Windows-10-10.0.19045-SP0

[필수 패키지]
[정상] requests: 2.34.2
[정상] python-dotenv: 1.2.4
[정상] pandas: 3.1.1
[정상] ipykernel: 7.4.0

[환경변수]
.env 파일          : 존재
.env 로딩 시도     : 완료
Client ID 변수    : 확인되지 않음
Client Secret 변수: 확인되지 않음
※ 인증키 값 자체는 보안을 위해 출력하지 않습니다.

[파일 및 폴더]
기존 원본 JSON     : C:\Users\user\Desktop\KDT_2026\프로젝트\03_융합_프로젝트\01_DB\data\raw\naver_trend_raw_20261119_215045.json
JSON 존재 여부     : 존재
data/raw           : 존재
data/processed     : 없음
logs               : 없음

[최종 점검표]
[정상] Python 버전 3.12.10
[정상] Jupyter 커널이 .venv 사용
[정상] 필수 패키지 import
[정상] .env 파일 존재
[확인 필요] 인증정보 2종 로딩
[정상] 기존 원본 JSON 존재
[정상] data/raw 폴더 존재
[확인 필요] data/processed 폴더 존재
[확인 필요] logs 폴더 존재
[정상] .env Git 제외 설정
[정상] .venv Git 제외 설정

확인 필요한 항목: 3개
- 인증정보 2종 로딩
- data/processed 폴더 존재
- logs

In [2]:
# Jupyter 커널 연결 확인

import sys
from pathlib import Path

python_path = Path(sys.executable).resolve()
uses_dot_venv = any(
    part.lower() == ".venv"
    for part in python_path.parts
)

print(f"현재 Python 경로: {python_path}")
print(f".venv 연결 여부  : {'정상' if uses_dot_venv else '확인 필요'}")

현재 Python 경로: C:\Users\user\Desktop\KDT_2026\프로젝트\03_융합_프로젝트\01_DB\.venv\Scripts\python.exe
.venv 연결 여부  : 정상


In [5]:
from dotenv import dotenv_values

env = dotenv_values(".env")

print("환경변수 이름:", list(env.keys()))
print("모든 값 설정:", all(bool(value) for value in env.values()))

환경변수 이름: ['NAVER_API_CLIENT_ID', 'NAVER_API_CLIENT_SECRET']
모든 값 설정: True


In [6]:
# STEP 2. NAVER 검색어 트렌드 API 인증 및 연결 테스트

import os
import json
import requests

from pathlib import Path
from dotenv import load_dotenv


# --------------------------------------------------
# 1) .env 인증정보 불러오기
# --------------------------------------------------
project_root = Path.cwd().resolve()
env_path = project_root / ".env"

load_dotenv(
    dotenv_path=env_path,
    override=True,
)

client_id = os.getenv("NAVER_API_CLIENT_ID")
client_secret = os.getenv("NAVER_API_CLIENT_SECRET")

if not client_id or not client_secret:
    raise ValueError(
        ".env에서 NAVER_API_CLIENT_ID 또는 "
        "NAVER_API_CLIENT_SECRET을 불러오지 못했습니다."
    )


# --------------------------------------------------
# 2) API 주소와 인증 헤더 설정
# --------------------------------------------------
api_url = (
    "https://naverapihub.apigw.ntruss.com"
    "/search-trend/v1/search"
)

headers = {
    "X-NCP-APIGW-API-KEY-ID": client_id,
    "X-NCP-APIGW-API-KEY": client_secret,
    "Content-Type": "application/json",
}


# --------------------------------------------------
# 3) 연결 테스트용 요청 조건
# --------------------------------------------------
test_payload = {
    "startDate": "2025-01-01",
    "endDate": "2025-01-31",
    "timeUnit": "month",
    "keywordGroups": [
        {
            "groupName": "일상기록",
            "keywords": ["일상기록"],
        }
    ],
}


# --------------------------------------------------
# 4) API 요청 1회 실행
# --------------------------------------------------
print("=" * 60)
print("NAVER 검색어 트렌드 API 연결 테스트")
print("=" * 60)
print(f"API 주소  : {api_url}")
print("테스트 기간: 2025-01-01 ~ 2025-01-31")
print("테스트 검색어: 일상기록")
print("인증정보  : 로딩 완료")
print("-" * 60)

try:
    response = requests.post(
        url=api_url,
        headers=headers,
        json=test_payload,
        timeout=30,
    )

    print(f"HTTP 상태 코드: {response.status_code}")

    if response.status_code == 200:
        response_data = response.json()

        print("API 연결 및 인증: 정상")
        print(f"응답 시작일: {response_data.get('startDate')}")
        print(f"응답 종료일: {response_data.get('endDate')}")
        print(f"조회 단위  : {response_data.get('timeUnit')}")

        results = response_data.get("results", [])

        print(f"결과 그룹 수: {len(results)}")

        if results:
            first_result = results[0]
            trend_data = first_result.get("data", [])

            print(f"그룹 이름  : {first_result.get('title')}")
            print(f"검색어 목록: {first_result.get('keywords')}")
            print(f"반환 데이터 수: {len(trend_data)}")

            if trend_data:
                print("\n첫 번째 데이터:")
                print(
                    json.dumps(
                        trend_data[0],
                        ensure_ascii=False,
                        indent=2,
                    )
                )
            else:
                print(
                    "\n연결과 인증은 정상이지만 "
                    "해당 조건의 검색 데이터가 없습니다."
                )
        else:
            print(
                "\n연결과 인증은 정상이지만 "
                "results가 비어 있습니다."
            )

    elif response.status_code in (401, 403):
        print("API 연결 실패: 인증정보를 확인해야 합니다.")
        print(f"응답 내용: {response.text[:1000]}")

    else:
        print("API 요청 실패")
        print(f"응답 내용: {response.text[:1000]}")

except requests.exceptions.Timeout:
    print("API 요청 실패: 30초 안에 응답하지 않았습니다.")

except requests.exceptions.ConnectionError as error:
    print("API 요청 실패: 서버 또는 인터넷 연결을 확인하세요.")
    print(f"오류 유형: {type(error).__name__}")

except requests.exceptions.RequestException as error:
    print("API 요청 중 오류가 발생했습니다.")
    print(f"오류 유형: {type(error).__name__}")
    print(f"오류 내용: {error}")

except ValueError as error:
    print("API 응답을 JSON 형식으로 해석하지 못했습니다.")
    print(f"오류 내용: {error}")

print("=" * 60)

NAVER 검색어 트렌드 API 연결 테스트
API 주소  : https://naverapihub.apigw.ntruss.com/search-trend/v1/search
테스트 기간: 2025-01-01 ~ 2025-01-31
테스트 검색어: 일상기록
인증정보  : 로딩 완료
------------------------------------------------------------
HTTP 상태 코드: 200
API 연결 및 인증: 정상
응답 시작일: 2025-01-01
응답 종료일: 2025-01-31
조회 단위  : month
결과 그룹 수: 1
그룹 이름  : 일상기록
검색어 목록: ['일상기록']
반환 데이터 수: 1

첫 번째 데이터:
{
  "period": "2025-01-01",
  "ratio": 100
}


In [7]:
# STEP 3. 전체 수집 조건 정의

from itertools import product
from collections import Counter
from pprint import pprint


# --------------------------------------------------
# 1) 조회 기간 및 단위
# --------------------------------------------------
START_DATE = "2025-01-01"
END_DATE = "2026-10-31"
TIME_UNIT = "month"


# --------------------------------------------------
# 2) 수집 키워드 정의
# --------------------------------------------------
KEYWORDS = [
    {
        "category": "일상 기록",
        "keyword": "일상기록",
    },
    {
        "category": "일상 기록",
        "keyword": "사진일기",
    },
    {
        "category": "일상 기록",
        "keyword": "포토덤프",
    },
    {
        "category": "기록 보관",
        "keyword": "사진정리",
    },
    {
        "category": "기록 보관",
        "keyword": "디지털다이어리",
    },
    {
        "category": "기록 보관",
        "keyword": "여행기록",
    },
    {
        "category": "장소 탐색",
        "keyword": "동네산책",
    },
    {
        "category": "장소 탐색",
        "keyword": "동네탐방",
    },
    {
        "category": "장소 탐색",
        "keyword": "스탬프투어",
    },
    {
        "category": "타인의 흔적",
        "keyword": "전시방명록",
    },
    {
        "category": "타인의 흔적",
        "keyword": "카페방명록",
    },
]


# --------------------------------------------------
# 3) 성별 조건 정의
# --------------------------------------------------
# 전체 성별은 API 요청에서 gender 필드를 생략
GENDERS = [
    {
        "gender_code": None,
        "gender_label": "전체",
    },
    {
        "gender_code": "m",
        "gender_label": "남성",
    },
    {
        "gender_code": "f",
        "gender_label": "여성",
    },
]


# --------------------------------------------------
# 4) 연령대 조건 정의
# --------------------------------------------------
# 전체 연령은 API 요청에서 ages 필드를 생략
AGES = [
    {
        "age_code": None,
        "age_label": "전체",
    },
    {
        "age_code": "1",
        "age_label": "0~12세",
    },
    {
        "age_code": "2",
        "age_label": "13~18세",
    },
    {
        "age_code": "3",
        "age_label": "19~24세",
    },
    {
        "age_code": "4",
        "age_label": "25~29세",
    },
    {
        "age_code": "5",
        "age_label": "30~34세",
    },
    {
        "age_code": "6",
        "age_label": "35~39세",
    },
    {
        "age_code": "7",
        "age_label": "40~44세",
    },
    {
        "age_code": "8",
        "age_label": "45~49세",
    },
    {
        "age_code": "9",
        "age_label": "50~54세",
    },
    {
        "age_code": "10",
        "age_label": "55~59세",
    },
    {
        "age_code": "11",
        "age_label": "60세 이상",
    },
]


# --------------------------------------------------
# 5) 전체 수집 조건 396개 생성
# --------------------------------------------------
COLLECTION_CONDITIONS = []

combinations = product(
    KEYWORDS,
    GENDERS,
    AGES,
)

for index, (keyword_info, gender_info, age_info) in enumerate(
    combinations,
    start=1,
):
    condition = {
        "condition_id": f"C{index:03d}",
        "category": keyword_info["category"],
        "keyword": keyword_info["keyword"],
        "gender_code": gender_info["gender_code"],
        "gender_label": gender_info["gender_label"],
        "age_code": age_info["age_code"],
        "age_label": age_info["age_label"],
        "start_date": START_DATE,
        "end_date": END_DATE,
        "time_unit": TIME_UNIT,
    }

    # 각 요청의 ratio 정규화 범위를 구분하기 위한 식별자
    condition["normalization_scope_id"] = (
        f"{condition['keyword']}|"
        f"{condition['gender_code'] or 'all'}|"
        f"{condition['age_code'] or 'all'}|"
        f"{START_DATE}|{END_DATE}|{TIME_UNIT}"
    )

    COLLECTION_CONDITIONS.append(condition)


# --------------------------------------------------
# 6) 조건 검증
# --------------------------------------------------
expected_condition_count = (
    len(KEYWORDS)
    * len(GENDERS)
    * len(AGES)
)

condition_ids = [
    condition["condition_id"]
    for condition in COLLECTION_CONDITIONS
]

normalization_scope_ids = [
    condition["normalization_scope_id"]
    for condition in COLLECTION_CONDITIONS
]

keyword_counts = Counter(
    condition["keyword"]
    for condition in COLLECTION_CONDITIONS
)

assert len(KEYWORDS) == 11, "키워드는 11개여야 합니다."
assert len(GENDERS) == 3, "성별 조건은 3개여야 합니다."
assert len(AGES) == 12, "연령 조건은 전체 포함 12개여야 합니다."

assert expected_condition_count == 396, (
    "예상 요청 수는 396개여야 합니다."
)

assert len(COLLECTION_CONDITIONS) == 396, (
    "생성된 수집 조건이 396개가 아닙니다."
)

assert len(condition_ids) == len(set(condition_ids)), (
    "condition_id가 중복되었습니다."
)

assert len(normalization_scope_ids) == len(
    set(normalization_scope_ids)
), "정규화 범위 식별자가 중복되었습니다."

assert all(
    count == 36
    for count in keyword_counts.values()
), "키워드별 조건 수는 각각 36개여야 합니다."


# --------------------------------------------------
# 7) 결과 출력
# --------------------------------------------------
print("=" * 70)
print("STEP 3 수집 조건 정의 및 검증 결과")
print("=" * 70)

print(f"조회 기간      : {START_DATE} ~ {END_DATE}")
print(f"조회 단위      : {TIME_UNIT}")
print(f"키워드 수      : {len(KEYWORDS)}개")
print(f"성별 조건 수   : {len(GENDERS)}개")
print(f"연령 조건 수   : {len(AGES)}개")
print(f"전체 요청 조건 : {len(COLLECTION_CONDITIONS)}개")

print("\n[카테고리별 키워드]")
for keyword_info in KEYWORDS:
    print(
        f"- {keyword_info['category']}: "
        f"{keyword_info['keyword']}"
    )

print("\n[키워드별 생성 조건 수]")
for keyword, count in keyword_counts.items():
    print(f"- {keyword}: {count}개")

print("\n[첫 번째 조건]")
pprint(
    COLLECTION_CONDITIONS[0],
    sort_dicts=False,
)

print("\n[마지막 조건]")
pprint(
    COLLECTION_CONDITIONS[-1],
    sort_dicts=False,
)

print("\n모든 수집 조건 검증: 정상")
print("=" * 70)

STEP 3 수집 조건 정의 및 검증 결과
조회 기간      : 2025-01-01 ~ 2026-10-31
조회 단위      : month
키워드 수      : 11개
성별 조건 수   : 3개
연령 조건 수   : 12개
전체 요청 조건 : 396개

[카테고리별 키워드]
- 일상 기록: 일상기록
- 일상 기록: 사진일기
- 일상 기록: 포토덤프
- 기록 보관: 사진정리
- 기록 보관: 디지털다이어리
- 기록 보관: 여행기록
- 장소 탐색: 동네산책
- 장소 탐색: 동네탐방
- 장소 탐색: 스탬프투어
- 타인의 흔적: 전시방명록
- 타인의 흔적: 카페방명록

[키워드별 생성 조건 수]
- 일상기록: 36개
- 사진일기: 36개
- 포토덤프: 36개
- 사진정리: 36개
- 디지털다이어리: 36개
- 여행기록: 36개
- 동네산책: 36개
- 동네탐방: 36개
- 스탬프투어: 36개
- 전시방명록: 36개
- 카페방명록: 36개

[첫 번째 조건]
{'condition_id': 'C001',
 'category': '일상 기록',
 'keyword': '일상기록',
 'gender_code': None,
 'gender_label': '전체',
 'age_code': None,
 'age_label': '전체',
 'start_date': '2025-01-01',
 'end_date': '2026-10-31',
 'time_unit': 'month',
 'normalization_scope_id': '일상기록|all|all|2025-01-01|2026-10-31|month'}

[마지막 조건]
{'condition_id': 'C396',
 'category': '타인의 흔적',
 'keyword': '카페방명록',
 'gender_code': 'f',
 'gender_label': '여성',
 'age_code': '11',
 'age_label': '60세 이상',
 'start_date': '2025-01-01',
 'end_date': '2026-1

In [8]:
# STEP 4. API 요청 함수와 재시도 처리

import os
import time
import requests

from datetime import datetime, timezone, timedelta
from pathlib import Path
from dotenv import load_dotenv


# --------------------------------------------------
# 1) API 인증정보 준비
# --------------------------------------------------
project_root = Path.cwd().resolve()
env_path = project_root / ".env"

load_dotenv(
    dotenv_path=env_path,
    override=True,
)

client_id = os.getenv("NAVER_API_CLIENT_ID")
client_secret = os.getenv("NAVER_API_CLIENT_SECRET")

if not client_id or not client_secret:
    raise ValueError("NAVER API 인증정보를 불러오지 못했습니다.")

API_URL = (
    "https://naverapihub.apigw.ntruss.com"
    "/search-trend/v1/search"
)

API_HEADERS = {
    "X-NCP-APIGW-API-KEY-ID": client_id,
    "X-NCP-APIGW-API-KEY": client_secret,
    "Content-Type": "application/json",
}


# --------------------------------------------------
# 2) 재시도 설정
# --------------------------------------------------
MAX_RETRIES = 3
REQUEST_TIMEOUT = 30
RETRY_DELAY_SECONDS = 2

# 일시적 문제일 가능성이 있어 재시도할 HTTP 상태 코드
RETRYABLE_STATUS_CODES = {
    429,  # 요청 한도 초과
    500,  # 서버 내부 오류
    502,  # 게이트웨이 오류
    503,  # 서비스 사용 불가
    504,  # 게이트웨이 시간 초과
}

KST = timezone(timedelta(hours=9))


# --------------------------------------------------
# 3) API 요청 본문 생성 함수
# --------------------------------------------------
def build_request_payload(condition):
    """
    STEP 3에서 생성한 수집 조건을
    NAVER API 요청 본문으로 변환합니다.
    """

    payload = {
        "startDate": condition["start_date"],
        "endDate": condition["end_date"],
        "timeUnit": condition["time_unit"],
        "keywordGroups": [
            {
                "groupName": condition["keyword"],
                "keywords": [condition["keyword"]],
            }
        ],
    }

    # 전체 성별이면 gender 필드를 보내지 않음
    if condition["gender_code"] is not None:
        payload["gender"] = condition["gender_code"]

    # 전체 연령이면 ages 필드를 보내지 않음
    if condition["age_code"] is not None:
        payload["ages"] = [condition["age_code"]]

    return payload


# --------------------------------------------------
# 4) API 요청 및 재시도 함수
# --------------------------------------------------
def request_naver_trend(
    condition,
    max_retries=MAX_RETRIES,
    timeout=REQUEST_TIMEOUT,
    retry_delay=RETRY_DELAY_SECONDS,
):
    """
    단일 조건으로 API를 요청합니다.

    max_retries=3이면 최초 요청 이후
    최대 3번 추가 재시도합니다.
    """

    payload = build_request_payload(condition)
    max_attempts = max_retries + 1
    attempt_log = []

    for attempt in range(1, max_attempts + 1):
        requested_at = datetime.now(KST).isoformat(
            timespec="seconds"
        )

        try:
            response = requests.post(
                url=API_URL,
                headers=API_HEADERS,
                json=payload,
                timeout=timeout,
            )

            http_status = response.status_code

            attempt_log.append(
                {
                    "attempt": attempt,
                    "requested_at": requested_at,
                    "http_status": http_status,
                    "error_type": None,
                    "error_message": None,
                }
            )

            # 정상 응답
            if http_status == 200:
                try:
                    response_json = response.json()
                except ValueError as error:
                    return {
                        "condition_id": condition["condition_id"],
                        "condition": condition,
                        "request_payload": payload,
                        "request_status": "invalid_json",
                        "http_status": http_status,
                        "attempt_count": attempt,
                        "data_point_count": 0,
                        "response_json": None,
                        "error_type": type(error).__name__,
                        "error_message": str(error),
                        "attempt_log": attempt_log,
                    }

                results = response_json.get("results", [])

                data_point_count = sum(
                    len(result.get("data", []))
                    for result in results
                )

                if data_point_count > 0:
                    request_status = "success_with_data"
                else:
                    request_status = "success_empty"

                return {
                    "condition_id": condition["condition_id"],
                    "condition": condition,
                    "request_payload": payload,
                    "request_status": request_status,
                    "http_status": http_status,
                    "attempt_count": attempt,
                    "data_point_count": data_point_count,
                    "response_json": response_json,
                    "error_type": None,
                    "error_message": None,
                    "attempt_log": attempt_log,
                }

            # 재시도 가능한 HTTP 오류
            if http_status in RETRYABLE_STATUS_CODES:
                if attempt < max_attempts:
                    wait_seconds = retry_delay * (2 ** (attempt - 1))

                    print(
                        f"[{condition['condition_id']}] "
                        f"HTTP {http_status} - "
                        f"{wait_seconds}초 후 재시도"
                    )

                    time.sleep(wait_seconds)
                    continue

            # 재시도하지 않거나 재시도를 모두 사용한 HTTP 오류
            return {
                "condition_id": condition["condition_id"],
                "condition": condition,
                "request_payload": payload,
                "request_status": "http_error",
                "http_status": http_status,
                "attempt_count": attempt,
                "data_point_count": 0,
                "response_json": None,
                "error_type": f"HTTP_{http_status}",
                "error_message": response.text[:2000],
                "attempt_log": attempt_log,
            }

        except (
            requests.exceptions.Timeout,
            requests.exceptions.ConnectionError,
        ) as error:
            attempt_log.append(
                {
                    "attempt": attempt,
                    "requested_at": requested_at,
                    "http_status": None,
                    "error_type": type(error).__name__,
                    "error_message": str(error),
                }
            )

            if attempt < max_attempts:
                wait_seconds = retry_delay * (2 ** (attempt - 1))

                print(
                    f"[{condition['condition_id']}] "
                    f"{type(error).__name__} - "
                    f"{wait_seconds}초 후 재시도"
                )

                time.sleep(wait_seconds)
                continue

            return {
                "condition_id": condition["condition_id"],
                "condition": condition,
                "request_payload": payload,
                "request_status": "network_error",
                "http_status": None,
                "attempt_count": attempt,
                "data_point_count": 0,
                "response_json": None,
                "error_type": type(error).__name__,
                "error_message": str(error),
                "attempt_log": attempt_log,
            }

        except requests.exceptions.RequestException as error:
            attempt_log.append(
                {
                    "attempt": attempt,
                    "requested_at": requested_at,
                    "http_status": None,
                    "error_type": type(error).__name__,
                    "error_message": str(error),
                }
            )

            return {
                "condition_id": condition["condition_id"],
                "condition": condition,
                "request_payload": payload,
                "request_status": "request_error",
                "http_status": None,
                "attempt_count": attempt,
                "data_point_count": 0,
                "response_json": None,
                "error_type": type(error).__name__,
                "error_message": str(error),
                "attempt_log": attempt_log,
            }


# --------------------------------------------------
# 5) 첫 번째 조건 C001만 테스트
# --------------------------------------------------
test_condition = COLLECTION_CONDITIONS[0]
test_result = request_naver_trend(test_condition)

print("=" * 70)
print("STEP 4 API 요청 함수 테스트 결과")
print("=" * 70)

print(f"조건 ID       : {test_result['condition_id']}")
print(f"키워드        : {test_condition['keyword']}")
print(f"성별          : {test_condition['gender_label']}")
print(f"연령          : {test_condition['age_label']}")
print(f"요청 상태     : {test_result['request_status']}")
print(f"HTTP 상태 코드: {test_result['http_status']}")
print(f"요청 시도 횟수: {test_result['attempt_count']}")
print(f"반환 데이터 수: {test_result['data_point_count']}")
print(f"오류 유형     : {test_result['error_type']}")

if test_result["request_status"] in {
    "success_with_data",
    "success_empty",
}:
    print("API 요청 함수 테스트: 정상")
else:
    print("API 요청 함수 테스트: 확인 필요")

print("=" * 70)

STEP 4 API 요청 함수 테스트 결과
조건 ID       : C001
키워드        : 일상기록
성별          : 전체
연령          : 전체
요청 상태     : success_with_data
HTTP 상태 코드: 200
요청 시도 횟수: 1
반환 데이터 수: 22
오류 유형     : None
API 요청 함수 테스트: 정상


In [9]:
# STEP 5. 전체 396개 조건 자동 수집

import time

from collections import Counter
from datetime import datetime


# API 요청 사이의 기본 대기 시간
REQUEST_INTERVAL_SECONDS = 0.2


# --------------------------------------------------
# 1) 수집 결과 저장 공간 준비
# --------------------------------------------------
# 같은 커널에서 셀을 다시 실행하면 완료된 조건은 건너뜀
if "COLLECTION_RESULTS" not in globals():
    COLLECTION_RESULTS = []

completed_condition_ids = {
    result["condition_id"]
    for result in COLLECTION_RESULTS
}

pending_conditions = [
    condition
    for condition in COLLECTION_CONDITIONS
    if condition["condition_id"] not in completed_condition_ids
]

collection_started_at = datetime.now(KST).isoformat(
    timespec="seconds"
)

print("=" * 70)
print("STEP 5 전체 조건 자동 수집")
print("=" * 70)
print(f"전체 조건 수   : {len(COLLECTION_CONDITIONS)}개")
print(f"기존 완료 조건 : {len(completed_condition_ids)}개")
print(f"이번 수집 대상 : {len(pending_conditions)}개")
print(f"요청 간격      : {REQUEST_INTERVAL_SECONDS}초")
print(f"수집 시작 시각 : {collection_started_at}")
print("-" * 70)


# --------------------------------------------------
# 2) 전체 조건 순차 수집
# --------------------------------------------------
interrupted = False

try:
    for sequence, condition in enumerate(
        pending_conditions,
        start=1,
    ):
        try:
            result = request_naver_trend(condition)

        except Exception as error:
            # 함수 외부에서 예상하지 못한 오류가 발생해도
            # 전체 수집이 중단되지 않도록 기록
            result = {
                "condition_id": condition["condition_id"],
                "condition": condition,
                "request_payload": build_request_payload(condition),
                "request_status": "internal_error",
                "http_status": None,
                "attempt_count": 0,
                "data_point_count": 0,
                "response_json": None,
                "error_type": type(error).__name__,
                "error_message": str(error),
                "attempt_log": [],
            }

        COLLECTION_RESULTS.append(result)

        current_total = len(COLLECTION_RESULTS)
        status = result["request_status"]

        # 오류는 즉시 출력
        if status not in {
            "success_with_data",
            "success_empty",
        }:
            print(
                f"[오류] {condition['condition_id']} | "
                f"{condition['keyword']} | "
                f"{condition['gender_label']} | "
                f"{condition['age_label']} | "
                f"{status}"
            )

        # 10개마다 진행 상황 출력
        if (
            current_total % 10 == 0
            or current_total == len(COLLECTION_CONDITIONS)
        ):
            current_counts = Counter(
                item["request_status"]
                for item in COLLECTION_RESULTS
            )

            current_success = (
                current_counts["success_with_data"]
                + current_counts["success_empty"]
            )

            print(
                f"[진행] {current_total}/"
                f"{len(COLLECTION_CONDITIONS)} | "
                f"정상 {current_success} | "
                f"데이터 있음 "
                f"{current_counts['success_with_data']} | "
                f"빈 응답 "
                f"{current_counts['success_empty']}"
            )

        # 마지막 요청 뒤에는 불필요하게 기다리지 않음
        if sequence < len(pending_conditions):
            time.sleep(REQUEST_INTERVAL_SECONDS)

except KeyboardInterrupt:
    interrupted = True

    print("\n사용자에 의해 수집이 중단되었습니다.")
    print(
        "현재 커널을 유지한 상태에서 이 셀을 다시 실행하면 "
        "완료되지 않은 조건부터 이어서 수집합니다."
    )


# --------------------------------------------------
# 3) 결과 정렬 및 중복 검증
# --------------------------------------------------
condition_order = {
    condition["condition_id"]: index
    for index, condition in enumerate(
        COLLECTION_CONDITIONS,
        start=1,
    )
}

COLLECTION_RESULTS.sort(
    key=lambda result: condition_order.get(
        result["condition_id"],
        999999,
    )
)

collected_ids = [
    result["condition_id"]
    for result in COLLECTION_RESULTS
]

if len(collected_ids) != len(set(collected_ids)):
    raise ValueError("수집 결과에 중복된 condition_id가 있습니다.")


# --------------------------------------------------
# 4) 수집 결과 요약
# --------------------------------------------------
status_counts = Counter(
    result["request_status"]
    for result in COLLECTION_RESULTS
)

success_with_data_count = status_counts[
    "success_with_data"
]

success_empty_count = status_counts[
    "success_empty"
]

success_count = (
    success_with_data_count
    + success_empty_count
)

failure_count = (
    len(COLLECTION_RESULTS)
    - success_count
)

total_data_point_count = sum(
    result["data_point_count"]
    for result in COLLECTION_RESULTS
)

collection_finished_at = datetime.now(KST).isoformat(
    timespec="seconds"
)

COLLECTION_SUMMARY = {
    "started_at": collection_started_at,
    "finished_at": collection_finished_at,
    "expected_condition_count": len(
        COLLECTION_CONDITIONS
    ),
    "collected_condition_count": len(
        COLLECTION_RESULTS
    ),
    "success_count": success_count,
    "success_with_data_count": success_with_data_count,
    "success_empty_count": success_empty_count,
    "failure_count": failure_count,
    "total_data_point_count": total_data_point_count,
    "interrupted": interrupted,
    "status_counts": dict(status_counts),
}


# --------------------------------------------------
# 5) 최종 결과 출력
# --------------------------------------------------
print("\n" + "=" * 70)
print("STEP 5 전체 수집 결과")
print("=" * 70)

print(
    f"예상 요청 조건 : "
    f"{COLLECTION_SUMMARY['expected_condition_count']}개"
)

print(
    f"수집 완료 조건 : "
    f"{COLLECTION_SUMMARY['collected_condition_count']}개"
)

print(
    f"정상 응답      : "
    f"{COLLECTION_SUMMARY['success_count']}개"
)

print(
    f"데이터 있음    : "
    f"{COLLECTION_SUMMARY['success_with_data_count']}개"
)

print(
    f"빈 응답        : "
    f"{COLLECTION_SUMMARY['success_empty_count']}개"
)

print(
    f"최종 실패      : "
    f"{COLLECTION_SUMMARY['failure_count']}개"
)

print(
    f"반환 데이터 행 : "
    f"{COLLECTION_SUMMARY['total_data_point_count']}개"
)

print(
    f"수집 중단 여부 : "
    f"{COLLECTION_SUMMARY['interrupted']}"
)

print("-" * 70)

if (
    len(COLLECTION_RESULTS) == 396
    and success_count == 396
    and failure_count == 0
):
    print("전체 수집 상태: 정상")
elif interrupted:
    print("전체 수집 상태: 중단됨 - 같은 셀을 다시 실행하세요.")
else:
    print("전체 수집 상태: 일부 오류 확인 필요")

print("=" * 70)

STEP 5 전체 조건 자동 수집
전체 조건 수   : 396개
기존 완료 조건 : 0개
이번 수집 대상 : 396개
요청 간격      : 0.2초
수집 시작 시각 : 2026-11-20T19:23:28+09:00
----------------------------------------------------------------------
[진행] 10/396 | 정상 10 | 데이터 있음 10 | 빈 응답 0
[진행] 20/396 | 정상 20 | 데이터 있음 20 | 빈 응답 0
[진행] 30/396 | 정상 30 | 데이터 있음 30 | 빈 응답 0
[진행] 40/396 | 정상 40 | 데이터 있음 40 | 빈 응답 0
[진행] 50/396 | 정상 50 | 데이터 있음 50 | 빈 응답 0
[진행] 60/396 | 정상 60 | 데이터 있음 60 | 빈 응답 0
[진행] 70/396 | 정상 70 | 데이터 있음 70 | 빈 응답 0
[진행] 80/396 | 정상 80 | 데이터 있음 80 | 빈 응답 0
[진행] 90/396 | 정상 90 | 데이터 있음 89 | 빈 응답 1
[진행] 100/396 | 정상 100 | 데이터 있음 99 | 빈 응답 1
[진행] 110/396 | 정상 110 | 데이터 있음 109 | 빈 응답 1
[진행] 120/396 | 정상 120 | 데이터 있음 119 | 빈 응답 1
[진행] 130/396 | 정상 130 | 데이터 있음 128 | 빈 응답 2
[진행] 140/396 | 정상 140 | 데이터 있음 138 | 빈 응답 2
[진행] 150/396 | 정상 150 | 데이터 있음 148 | 빈 응답 2
[진행] 160/396 | 정상 160 | 데이터 있음 158 | 빈 응답 2
[진행] 170/396 | 정상 170 | 데이터 있음 168 | 빈 응답 2
[진행] 180/396 | 정상 180 | 데이터 있음 178 | 빈 응답 2
[진행] 190/396 | 정상 190 | 데이터 있음 188 | 빈 응답 2


In [10]:
# STEP 6. API 원본 JSON 저장 및 검증

import json
import hashlib

from datetime import datetime
from pathlib import Path


# --------------------------------------------------
# 1) 저장 전 전체 수집 상태 확인
# --------------------------------------------------
if len(COLLECTION_RESULTS) != 396:
    raise ValueError(
        f"수집 결과가 396개가 아닙니다: "
        f"{len(COLLECTION_RESULTS)}개"
    )

if COLLECTION_SUMMARY["failure_count"] != 0:
    raise ValueError(
        "최종 실패 요청이 있어 원본 JSON 저장을 중단합니다."
    )

condition_ids = [
    result["condition_id"]
    for result in COLLECTION_RESULTS
]

if len(condition_ids) != len(set(condition_ids)):
    raise ValueError(
        "수집 결과에 중복된 condition_id가 있습니다."
    )


# --------------------------------------------------
# 2) 저장 경로 및 새 파일명 설정
# --------------------------------------------------
project_root = Path.cwd().resolve()
raw_dir = project_root / "data" / "raw"

# data/raw이 이미 있으면 아무 변화 없음
raw_dir.mkdir(
    parents=True,
    exist_ok=True,
)

backup_json_path = (
    raw_dir / "naver_trend_raw_20261119_215045.json"
)

saved_at = datetime.now(KST)
timestamp = saved_at.strftime("%Y%m%d_%H%M%S")

new_raw_json_path = (
    raw_dir / f"naver_trend_raw_{timestamp}.json"
)

if new_raw_json_path.exists():
    raise FileExistsError(
        f"같은 이름의 파일이 이미 존재합니다: "
        f"{new_raw_json_path}"
    )


# --------------------------------------------------
# 3) 원본 JSON 저장 구조 구성
# --------------------------------------------------
raw_json_data = {
    "metadata": {
        "schema_version": "1.0",
        "source": "NAVER API HUB Search Trend",
        "api_url": API_URL,
        "saved_at": saved_at.isoformat(
            timespec="seconds"
        ),
        "collection_period": {
            "start_date": START_DATE,
            "end_date": END_DATE,
            "time_unit": TIME_UNIT,
        },
        "keyword_count": len(KEYWORDS),
        "gender_condition_count": len(GENDERS),
        "age_condition_count": len(AGES),
        "expected_request_count": 396,
        "keywords": KEYWORDS,
        "genders": GENDERS,
        "ages": AGES,
        "normalization_note": (
            "ratio는 각 API 요청 조건과 조회 기간 안에서 "
            "최댓값을 100으로 계산한 상대 지수이다. "
            "서로 다른 normalization_scope_id의 ratio를 "
            "실제 검색량처럼 직접 비교하면 안 된다."
        ),
        "collection_summary": COLLECTION_SUMMARY,
    },
    "results": COLLECTION_RESULTS,
}


# --------------------------------------------------
# 4) 새 원본 JSON 저장
# --------------------------------------------------
# x 모드는 기존 파일을 절대 덮어쓰지 않음
with new_raw_json_path.open(
    mode="x",
    encoding="utf-8",
) as file:
    json.dump(
        raw_json_data,
        file,
        ensure_ascii=False,
        indent=2,
    )


# --------------------------------------------------
# 5) 저장된 JSON 다시 불러와 검증
# --------------------------------------------------
with new_raw_json_path.open(
    mode="r",
    encoding="utf-8",
) as file:
    verified_json_data = json.load(file)

verified_results = verified_json_data.get(
    "results",
    [],
)

verified_condition_ids = [
    result.get("condition_id")
    for result in verified_results
]

verified_summary = (
    verified_json_data
    .get("metadata", {})
    .get("collection_summary", {})
)

file_size_bytes = new_raw_json_path.stat().st_size

with new_raw_json_path.open("rb") as file:
    sha256_hash = hashlib.sha256(
        file.read()
    ).hexdigest()


# --------------------------------------------------
# 6) 저장 검증 조건
# --------------------------------------------------
validation_checks = {
    "새 JSON 파일 존재": new_raw_json_path.is_file(),
    "저장 결과 396개": len(verified_results) == 396,
    "condition_id 중복 없음": (
        len(verified_condition_ids)
        == len(set(verified_condition_ids))
    ),
    "저장된 실패 요청 0개": (
        verified_summary.get("failure_count") == 0
    ),
    "기존 백업 JSON 보존": backup_json_path.is_file(),
    "기존 파일과 새 파일 구분": (
        backup_json_path.resolve()
        != new_raw_json_path.resolve()
    ),
}

all_valid = all(validation_checks.values())

# 이후 단계에서 사용할 새 원본 JSON 경로
RAW_JSON_PATH = new_raw_json_path


# --------------------------------------------------
# 7) 결과 출력
# --------------------------------------------------
print("=" * 70)
print("STEP 6 원본 JSON 저장 및 검증 결과")
print("=" * 70)

print(f"새 원본 JSON : {new_raw_json_path}")
print(f"기존 백업 JSON: {backup_json_path}")
print(f"파일 크기     : {file_size_bytes:,} bytes")
print(f"SHA-256       : {sha256_hash}")

print("\n[저장 검증]")
for check_name, passed in validation_checks.items():
    status = "정상" if passed else "확인 필요"
    print(f"[{status}] {check_name}")

print("-" * 70)

if all_valid:
    print("원본 JSON 저장 상태: 정상")
    print(
        "이제 API를 다시 호출하지 않고 "
        "새 JSON에서 정제 작업을 재개할 수 있습니다."
    )
else:
    print("원본 JSON 저장 상태: 확인 필요")

print("=" * 70)

STEP 6 원본 JSON 저장 및 검증 결과
새 원본 JSON : C:\Users\user\Desktop\KDT_2026\프로젝트\03_융합_프로젝트\01_DB\data\raw\naver_trend_raw_20261120_192943.json
기존 백업 JSON: C:\Users\user\Desktop\KDT_2026\프로젝트\03_융합_프로젝트\01_DB\data\raw\naver_trend_raw_20261119_215045.json
파일 크기     : 1,289,633 bytes
SHA-256       : 20ebd5d89197717d58f026d742baeadc1841cda49b8f8d4cb46867f3379fd87a

[저장 검증]
[정상] 새 JSON 파일 존재
[정상] 저장 결과 396개
[정상] condition_id 중복 없음
[정상] 저장된 실패 요청 0개
[확인 필요] 기존 백업 JSON 보존
[정상] 기존 파일과 새 파일 구분
----------------------------------------------------------------------
원본 JSON 저장 상태: 확인 필요


In [11]:
# STEP 7. 원본 JSON 불러오기 및 분석용 데이터 정제

import json
import pandas as pd

from pathlib import Path


# --------------------------------------------------
# 1) 새 원본 JSON 경로 지정
# --------------------------------------------------
project_root = Path.cwd().resolve()

raw_json_path = (
    project_root
    / "data"
    / "raw"
    / "naver_trend_raw_20261120_192943.json"
)

if not raw_json_path.is_file():
    raise FileNotFoundError(
        f"원본 JSON을 찾을 수 없습니다: {raw_json_path}"
    )


# --------------------------------------------------
# 2) 원본 JSON 불러오기
# --------------------------------------------------
with raw_json_path.open(
    mode="r",
    encoding="utf-8",
) as file:
    loaded_raw_data = json.load(file)

metadata = loaded_raw_data.get("metadata", {})
request_results = loaded_raw_data.get("results", [])

if len(request_results) != 396:
    raise ValueError(
        f"원본 JSON의 요청 결과가 396개가 아닙니다: "
        f"{len(request_results)}개"
    )


# --------------------------------------------------
# 3) API 응답을 행 단위로 변환
# --------------------------------------------------
rows = []

for request_result in request_results:
    condition = request_result.get("condition", {})
    response_json = request_result.get("response_json") or {}
    response_results = response_json.get("results", [])

    # success_empty 조건에서는 반복할 데이터가 없으므로
    # 행을 임의로 생성하지 않음
    for response_group in response_results:
        returned_title = response_group.get("title")
        returned_keywords = response_group.get(
            "keywords",
            [],
        )
        trend_data = response_group.get("data", [])

        for data_point in trend_data:
            row = {
                "condition_id": request_result.get(
                    "condition_id"
                ),
                "normalization_scope_id": condition.get(
                    "normalization_scope_id"
                ),
                "category": condition.get("category"),
                "keyword": condition.get("keyword"),
                "returned_title": returned_title,
                "returned_keywords": ",".join(
                    returned_keywords
                ),
                "gender_code": condition.get(
                    "gender_code"
                ),
                "gender_label": condition.get(
                    "gender_label"
                ),
                "age_code": condition.get("age_code"),
                "age_label": condition.get("age_label"),
                "period": data_point.get("period"),
                "ratio": data_point.get("ratio"),
                "start_date": condition.get(
                    "start_date"
                ),
                "end_date": condition.get("end_date"),
                "time_unit": condition.get(
                    "time_unit"
                ),
                "request_status": request_result.get(
                    "request_status"
                ),
                "http_status": request_result.get(
                    "http_status"
                ),
                "attempt_count": request_result.get(
                    "attempt_count"
                ),
                "source_file": raw_json_path.name,
            }

            rows.append(row)

raw_trend_df = pd.DataFrame(rows)

if raw_trend_df.empty:
    raise ValueError(
        "원본 JSON에서 분석할 데이터 행을 찾지 못했습니다."
    )

original_row_count = len(raw_trend_df)


# --------------------------------------------------
# 4) 데이터 타입 정리
# --------------------------------------------------
date_columns = [
    "period",
    "start_date",
    "end_date",
]

for column in date_columns:
    raw_trend_df[column] = pd.to_datetime(
        raw_trend_df[column],
        errors="coerce",
    )

raw_trend_df["ratio"] = pd.to_numeric(
    raw_trend_df["ratio"],
    errors="coerce",
)

raw_trend_df["http_status"] = pd.to_numeric(
    raw_trend_df["http_status"],
    errors="coerce",
).astype("Int64")

raw_trend_df["attempt_count"] = pd.to_numeric(
    raw_trend_df["attempt_count"],
    errors="coerce",
).astype("Int64")

string_columns = [
    "condition_id",
    "normalization_scope_id",
    "category",
    "keyword",
    "returned_title",
    "returned_keywords",
    "gender_code",
    "gender_label",
    "age_code",
    "age_label",
    "time_unit",
    "request_status",
    "source_file",
]

for column in string_columns:
    raw_trend_df[column] = (
        raw_trend_df[column]
        .astype("string")
    )


# --------------------------------------------------
# 5) 핵심값이 잘못된 행 분리
# --------------------------------------------------
invalid_required_mask = (
    raw_trend_df["condition_id"].isna()
    | raw_trend_df["keyword"].isna()
    | raw_trend_df["period"].isna()
    | raw_trend_df["ratio"].isna()
)

invalid_rows_df = (
    raw_trend_df.loc[invalid_required_mask]
    .copy()
)

trend_df = (
    raw_trend_df.loc[~invalid_required_mask]
    .copy()
)


# --------------------------------------------------
# 6) 중복 확인 및 제거
# --------------------------------------------------
duplicate_key = [
    "condition_id",
    "period",
]

duplicate_mask = trend_df.duplicated(
    subset=duplicate_key,
    keep="first",
)

duplicate_rows_df = (
    trend_df.loc[duplicate_mask]
    .copy()
)

trend_df = (
    trend_df.loc[~duplicate_mask]
    .copy()
)


# --------------------------------------------------
# 7) 정렬 및 최종 타입 정리
# --------------------------------------------------
trend_df = (
    trend_df
    .sort_values(
        by=[
            "condition_id",
            "period",
        ]
    )
    .reset_index(drop=True)
)

trend_df["ratio"] = trend_df["ratio"].astype(
    "float64"
)

cleaned_row_count = len(trend_df)


# --------------------------------------------------
# 8) 정제 결과 출력
# --------------------------------------------------
print("=" * 70)
print("STEP 7 원본 JSON 로딩 및 정제 결과")
print("=" * 70)

print(f"사용 원본 파일   : {raw_json_path.name}")
print(f"API 요청 결과 수 : {len(request_results)}개")
print(f"원본 데이터 행   : {original_row_count:,}개")
print(f"핵심값 오류 행   : {len(invalid_rows_df):,}개")
print(f"중복 제거 행     : {len(duplicate_rows_df):,}개")
print(f"최종 정제 행     : {cleaned_row_count:,}개")

print("\n[최종 데이터 타입]")
print(trend_df.dtypes)

print("\n[정제 데이터 미리보기]")
display(trend_df.head())

print("\n[처리 원칙]")
print("- API가 반환하지 않은 월은 행을 생성하지 않음")
print("- 결측 월과 결측 ratio를 0으로 대체하지 않음")
print("- 원본 JSON은 수정하지 않음")

print("=" * 70)

STEP 7 원본 JSON 로딩 및 정제 결과
사용 원본 파일   : naver_trend_raw_20261120_192943.json
API 요청 결과 수 : 396개
원본 데이터 행   : 5,801개
핵심값 오류 행   : 0개
중복 제거 행     : 0개
최종 정제 행     : 5,801개

[최종 데이터 타입]
condition_id                      string
normalization_scope_id            string
category                          string
keyword                           string
returned_title                    string
returned_keywords                 string
gender_code                       string
gender_label                      string
age_code                          string
age_label                         string
period                    datetime64[us]
ratio                            float64
start_date                datetime64[us]
end_date                  datetime64[us]
time_unit                         string
request_status                    string
http_status                        Int64
attempt_count                      Int64
source_file                       string
dtype: object

[정제 데이터 미리보기]


,condition_id,normalization_scope_id,category,keyword,returned_title,returned_keywords,gender_code,gender_label,age_code,age_label,period,ratio,start_date,end_date,time_unit,request_status,http_status,attempt_count,source_file
0,C001,일상기록|all|all|2025-01-01|2026-10-31|month,일상 기록,일상기록,일상기록,일상기록,<NA>,전체,<NA>,전체,2025-01-01,100.00000,2025-01-01,2026-10-31,month,success_with_data,200,1,naver_trend_raw_20261120_192943.json
1,C001,일상기록|all|all|2025-01-01|2026-10-31|month,일상 기록,일상기록,일상기록,일상기록,<NA>,전체,<NA>,전체,2025-02-01,82.58706,2025-01-01,2026-10-31,month,success_with_data,200,1,naver_trend_raw_20261120_192943.json
2,C001,일상기록|all|all|2025-01-01|2026-10-31|month,일상 기록,일상기록,일상기록,일상기록,<NA>,전체,<NA>,전체,2025-03-01,88.55721,2025-01-01,2026-10-31,month,success_with_data,200,1,naver_trend_raw_20261120_192943.json
3,C001,일상기록|all|all|2025-01-01|2026-10-31|month,일상 기록,일상기록,일상기록,일상기록,<NA>,전체,<NA>,전체,2025-04-01,80.76285,2025-01-01,2026-10-31,month,success_with_data,200,1,naver_trend_raw_20261120_192943.json
4,C001,일상기록|all|all|2025-01-01|2026-10-31|month,일상 기록,일상기록,일상기록,일상기록,<NA>,전체,<NA>,전체,2025-05-01,69.98341,2025-01-01,2026-10-31,month,success_with_data,200,1,naver_trend_raw_20261120_192943.json



[처리 원칙]
- API가 반환하지 않은 월은 행을 생성하지 않음
- 결측 월과 결측 ratio를 0으로 대체하지 않음
- 원본 JSON은 수정하지 않음


In [12]:
# age_code 상태 확인

print("[age_code별 행 수]")
print(
    trend_df["age_code"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n[확인 결과]")
print(f"전체 행 수       : {len(trend_df):,}")
print(f"null 행 수       : {trend_df['age_code'].isna().sum():,}")
print(f"null 아닌 행 수  : {trend_df['age_code'].notna().sum():,}")
print(f"전부 null인지    : {trend_df['age_code'].isna().all()}")

age_inconsistent_mask = (
    (
        trend_df["age_label"].eq("전체")
        & trend_df["age_code"].notna()
    )
    |
    (
        trend_df["age_label"].ne("전체")
        & trend_df["age_code"].isna()
    )
)

print(
    f"코드·라벨 불일치 : "
    f"{age_inconsistent_mask.sum():,}건"
)

[age_code별 행 수]
age_code
1       143
10      481
11      485
2       447
3       515
4       526
5       531
6       544
7       532
8       522
9       505
<NA>    570
Name: count, dtype: Int64

[확인 결과]
전체 행 수       : 5,801
null 행 수       : 570
null 아닌 행 수  : 5,231
전부 null인지    : False
코드·라벨 불일치 : 0건


In [13]:
# STEP 8. 데이터 품질 검증

from collections import Counter

import pandas as pd


# --------------------------------------------------
# 1) 원본 요청 상태 집계
# --------------------------------------------------
request_status_counts = Counter(
    result.get("request_status")
    for result in request_results
)

success_with_data_count = request_status_counts[
    "success_with_data"
]

success_empty_count = request_status_counts[
    "success_empty"
]

success_count = (
    success_with_data_count
    + success_empty_count
)

failure_count = (
    len(request_results)
    - success_count
)

raw_data_point_count = sum(
    int(result.get("data_point_count", 0))
    for result in request_results
)


# --------------------------------------------------
# 2) 메타데이터 기준값 준비
# --------------------------------------------------
keyword_metadata = metadata.get("keywords", [])
gender_metadata = metadata.get("genders", [])
age_metadata = metadata.get("ages", [])

valid_keywords = {
    item["keyword"]
    for item in keyword_metadata
}

category_by_keyword = {
    item["keyword"]: item["category"]
    for item in keyword_metadata
}

gender_label_by_code = {
    item.get("gender_code") or "__ALL__":
    item["gender_label"]
    for item in gender_metadata
}

age_label_by_code = {
    item.get("age_code") or "__ALL__":
    item["age_label"]
    for item in age_metadata
}

collection_period = metadata.get(
    "collection_period",
    {},
)

expected_start_date = pd.Timestamp(
    collection_period["start_date"]
)

expected_end_date = pd.Timestamp(
    collection_period["end_date"]
)

expected_months = pd.date_range(
    start=expected_start_date,
    end=expected_end_date,
    freq="MS",
)

expected_month_count = len(expected_months)


# --------------------------------------------------
# 3) 코드와 라벨 일관성 확인
# --------------------------------------------------
expected_categories = trend_df["keyword"].map(
    category_by_keyword
)

gender_keys = (
    trend_df["gender_code"]
    .fillna("__ALL__")
)

expected_gender_labels = gender_keys.map(
    gender_label_by_code
)

age_keys = (
    trend_df["age_code"]
    .fillna("__ALL__")
)

expected_age_labels = age_keys.map(
    age_label_by_code
)


# --------------------------------------------------
# 4) 요청별 ratio 최대값 확인
# --------------------------------------------------
max_ratio_by_condition = (
    trend_df
    .groupby("condition_id")["ratio"]
    .max()
)

invalid_max_ratio_conditions = (
    max_ratio_by_condition[
        (max_ratio_by_condition - 100).abs() > 1e-9
    ]
)


# --------------------------------------------------
# 5) 조건별 월 반환 현황
# --------------------------------------------------
coverage_rows = []

for result in request_results:
    condition = result.get("condition", {})

    returned_month_count = int(
        result.get("data_point_count", 0)
    )

    coverage_rows.append(
        {
            "condition_id": result.get(
                "condition_id"
            ),
            "keyword": condition.get("keyword"),
            "gender_label": condition.get(
                "gender_label"
            ),
            "age_label": condition.get(
                "age_label"
            ),
            "request_status": result.get(
                "request_status"
            ),
            "expected_month_count": (
                expected_month_count
            ),
            "returned_month_count": (
                returned_month_count
            ),
            "unreturned_month_count": max(
                expected_month_count
                - returned_month_count,
                0,
            ),
        }
    )

condition_coverage_df = pd.DataFrame(
    coverage_rows
)

full_coverage_count = int(
    (
        condition_coverage_df[
            "returned_month_count"
        ]
        == expected_month_count
    ).sum()
)

partial_coverage_count = int(
    (
        (
            condition_coverage_df[
                "returned_month_count"
            ] > 0
        )
        & (
            condition_coverage_df[
                "returned_month_count"
            ] < expected_month_count
        )
    ).sum()
)

zero_coverage_count = int(
    (
        condition_coverage_df[
            "returned_month_count"
        ] == 0
    ).sum()
)

total_unreturned_month_count = int(
    condition_coverage_df[
        "unreturned_month_count"
    ].sum()
)


# --------------------------------------------------
# 6) 빈 응답 조건에 행이 생성되지 않았는지 확인
# --------------------------------------------------
empty_condition_ids = {
    result["condition_id"]
    for result in request_results
    if result.get("request_status")
    == "success_empty"
}

data_condition_ids = set(
    trend_df["condition_id"].dropna()
)

empty_conditions_with_rows = (
    empty_condition_ids
    & data_condition_ids
)


# --------------------------------------------------
# 7) 품질 검증 항목
# --------------------------------------------------
quality_checks = {
    "원본 요청 결과 396개": (
        len(request_results) == 396
    ),
    "condition_id 중복 없음": (
        len({
            result.get("condition_id")
            for result in request_results
        })
        == len(request_results)
    ),
    "API 최종 실패 0개": (
        failure_count == 0
    ),
    "원본 응답 행 수 일치": (
        raw_data_point_count
        == len(raw_trend_df)
    ),
    "핵심값 오류 0개": (
        len(invalid_rows_df) == 0
    ),
    "중복 데이터 0개": (
        len(duplicate_rows_df) == 0
    ),
    "키워드 11개 기준과 일치": (
        trend_df["keyword"]
        .isin(valid_keywords)
        .all()
    ),
    "키워드와 카테고리 일치": (
        expected_categories
        .eq(trend_df["category"])
        .all()
    ),
    "성별 코드와 라벨 일치": (
        expected_gender_labels
        .eq(trend_df["gender_label"])
        .all()
    ),
    "연령 코드와 라벨 일치": (
        expected_age_labels
        .eq(trend_df["age_label"])
        .all()
    ),
    "날짜 결측 없음": (
        trend_df["period"].notna().all()
    ),
    "날짜가 조회 범위 안에 있음": (
        trend_df["period"]
        .between(
            expected_start_date,
            expected_end_date,
            inclusive="both",
        )
        .all()
    ),
    "모든 period가 월 시작일": (
        trend_df["period"]
        .dt.is_month_start
        .all()
    ),
    "ratio 결측 없음": (
        trend_df["ratio"].notna().all()
    ),
    "ratio가 0~100 범위": (
        trend_df["ratio"]
        .between(
            0,
            100,
            inclusive="both",
        )
        .all()
    ),
    "데이터 조건별 최대 ratio가 100": (
        len(invalid_max_ratio_conditions) == 0
    ),
    "데이터 조건 수와 API 집계 일치": (
        trend_df["condition_id"].nunique()
        == success_with_data_count
    ),
    "빈 응답 조건에 생성된 행 없음": (
        len(empty_conditions_with_rows) == 0
    ),
    "요청 상태가 데이터 있음으로 일치": (
        trend_df["request_status"]
        .eq("success_with_data")
        .all()
    ),
    "조회 단위가 모두 month": (
        trend_df["time_unit"]
        .eq("month")
        .all()
    ),
    "반환 제목과 요청 키워드 일치": (
        trend_df["returned_title"]
        .eq(trend_df["keyword"])
        .all()
    ),
    "조건별 정규화 범위가 하나": (
        trend_df
        .groupby("condition_id")[
            "normalization_scope_id"
        ]
        .nunique()
        .le(1)
        .all()
    ),
    "정규화 범위별 조건이 하나": (
        trend_df
        .groupby("normalization_scope_id")[
            "condition_id"
        ]
        .nunique()
        .le(1)
        .all()
    ),
    "조건별 반환 월이 최대 22개": (
        condition_coverage_df[
            "returned_month_count"
        ]
        .le(expected_month_count)
        .all()
    ),
}

all_quality_checks_passed = all(
    bool(result)
    for result in quality_checks.values()
)

failed_quality_checks = [
    check_name
    for check_name, passed in quality_checks.items()
    if not bool(passed)
]


# --------------------------------------------------
# 8) 이후 저장 단계에서 사용할 품질 보고서
# --------------------------------------------------
QUALITY_REPORT = {
    "request_count": len(request_results),
    "success_count": success_count,
    "success_with_data_count": (
        success_with_data_count
    ),
    "success_empty_count": success_empty_count,
    "failure_count": failure_count,
    "cleaned_row_count": len(trend_df),
    "expected_month_count_per_condition": (
        expected_month_count
    ),
    "full_coverage_condition_count": (
        full_coverage_count
    ),
    "partial_coverage_condition_count": (
        partial_coverage_count
    ),
    "zero_coverage_condition_count": (
        zero_coverage_count
    ),
    "total_unreturned_month_count": (
        total_unreturned_month_count
    ),
    "all_checks_passed": (
        all_quality_checks_passed
    ),
    "failed_checks": failed_quality_checks,
}


# --------------------------------------------------
# 9) 결과 출력
# --------------------------------------------------
print("=" * 70)
print("STEP 8 데이터 품질 검증 결과")
print("=" * 70)

print(f"전체 API 요청      : {len(request_results)}개")
print(f"정상 응답          : {success_count}개")
print(f"데이터 있음        : {success_with_data_count}개")
print(f"빈 응답            : {success_empty_count}개")
print(f"최종 실패          : {failure_count}개")
print(f"최종 분석 데이터   : {len(trend_df):,}행")

print("\n[월별 데이터 반환 현황]")
print(f"조건별 예상 월     : {expected_month_count}개월")
print(f"전체 월 반환 조건  : {full_coverage_count}개")
print(f"일부 월 반환 조건  : {partial_coverage_count}개")
print(f"데이터 없는 조건   : {zero_coverage_count}개")
print(
    f"API 미반환 조건-월 : "
    f"{total_unreturned_month_count:,}개"
)

print("\n[품질 검증 항목]")
for check_name, passed in quality_checks.items():
    status = "정상" if bool(passed) else "확인 필요"
    print(f"[{status}] {check_name}")

print("-" * 70)

if all_quality_checks_passed:
    print("데이터 품질 검증: 정상")
else:
    print("데이터 품질 검증: 확인 필요")
    print("실패 항목:")
    for check_name in failed_quality_checks:
        print(f"- {check_name}")

print("\n※ API 미반환 월은 오류나 0으로 처리하지 않습니다.")
print("※ 서로 다른 정규화 범위의 ratio는 직접 비교하지 않습니다.")

print("=" * 70)

STEP 8 데이터 품질 검증 결과
전체 API 요청      : 396개
정상 응답          : 396개
데이터 있음        : 342개
빈 응답            : 54개
최종 실패          : 0개
최종 분석 데이터   : 5,801행

[월별 데이터 반환 현황]
조건별 예상 월     : 22개월
전체 월 반환 조건  : 196개
일부 월 반환 조건  : 146개
데이터 없는 조건   : 54개
API 미반환 조건-월 : 2,911개

[품질 검증 항목]
[정상] 원본 요청 결과 396개
[정상] condition_id 중복 없음
[정상] API 최종 실패 0개
[정상] 원본 응답 행 수 일치
[정상] 핵심값 오류 0개
[정상] 중복 데이터 0개
[정상] 키워드 11개 기준과 일치
[정상] 키워드와 카테고리 일치
[정상] 성별 코드와 라벨 일치
[정상] 연령 코드와 라벨 일치
[정상] 날짜 결측 없음
[정상] 날짜가 조회 범위 안에 있음
[정상] 모든 period가 월 시작일
[정상] ratio 결측 없음
[정상] ratio가 0~100 범위
[정상] 데이터 조건별 최대 ratio가 100
[정상] 데이터 조건 수와 API 집계 일치
[정상] 빈 응답 조건에 생성된 행 없음
[정상] 요청 상태가 데이터 있음으로 일치
[정상] 조회 단위가 모두 month
[정상] 반환 제목과 요청 키워드 일치
[정상] 조건별 정규화 범위가 하나
[정상] 정규화 범위별 조건이 하나
[정상] 조건별 반환 월이 최대 22개
----------------------------------------------------------------------
데이터 품질 검증: 정상

※ API 미반환 월은 오류나 0으로 처리하지 않습니다.
※ 서로 다른 정규화 범위의 ratio는 직접 비교하지 않습니다.


In [14]:
# STEP 9. 분석용 CSV 저장 및 검증

import hashlib
import pandas as pd

from pathlib import Path


# --------------------------------------------------
# 1) 저장 전 품질 검증 결과 확인
# --------------------------------------------------
if not QUALITY_REPORT.get("all_checks_passed", False):
    raise ValueError(
        "STEP 8 품질 검증을 통과하지 못해 "
        "CSV 저장을 중단합니다."
    )

if len(trend_df) != 5801:
    print(
        f"참고: 현재 정제 행 수는 "
        f"{len(trend_df):,}개입니다."
    )


# --------------------------------------------------
# 2) CSV 내보내기용 데이터 복사
# --------------------------------------------------
# 원본 trend_df는 변경하지 않음
export_df = trend_df.copy()


# 전체 조건의 코드를 명시적인 all로 변환
export_df["gender_code"] = (
    export_df["gender_code"]
    .fillna("all")
)

export_df["age_code"] = (
    export_df["age_code"]
    .fillna("all")
)


# 날짜는 MySQL DATE와 호환되는 문자열로 변환
date_columns = [
    "period",
    "start_date",
    "end_date",
]

for column in date_columns:
    export_df[column] = (
        export_df[column]
        .dt.strftime("%Y-%m-%d")
    )


# --------------------------------------------------
# 3) 분석에 편리한 열 순서 지정
# --------------------------------------------------
export_columns = [
    "condition_id",
    "normalization_scope_id",
    "category",
    "keyword",
    "gender_code",
    "gender_label",
    "age_code",
    "age_label",
    "period",
    "ratio",
    "start_date",
    "end_date",
    "time_unit",
    "request_status",
    "http_status",
    "attempt_count",
    "returned_title",
    "returned_keywords",
    "source_file",
]

export_df = export_df[export_columns]


# --------------------------------------------------
# 4) 저장 경로와 파일명 지정
# --------------------------------------------------
processed_dir = (
    project_root
    / "data"
    / "processed"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True,
)

raw_timestamp = (
    raw_json_path
    .stem
    .removeprefix("naver_trend_raw_")
)

csv_path = (
    processed_dir
    / f"naver_trend_clean_{raw_timestamp}.csv"
)

if csv_path.exists():
    raise FileExistsError(
        f"CSV가 이미 존재합니다: {csv_path}"
    )


# --------------------------------------------------
# 5) CSV 저장
# --------------------------------------------------
# utf-8-sig: Excel에서 한글이 깨지지 않도록 저장
export_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig",
)


# --------------------------------------------------
# 6) 저장된 CSV 다시 불러오기
# --------------------------------------------------
verified_csv_df = pd.read_csv(
    csv_path,
    encoding="utf-8-sig",
    dtype={
        "condition_id": "string",
        "normalization_scope_id": "string",
        "gender_code": "string",
        "age_code": "string",
    },
)

verified_csv_df["period"] = pd.to_datetime(
    verified_csv_df["period"],
    errors="coerce",
)

verified_csv_df["ratio"] = pd.to_numeric(
    verified_csv_df["ratio"],
    errors="coerce",
)


# --------------------------------------------------
# 7) CSV 검증
# --------------------------------------------------
csv_duplicate_count = (
    verified_csv_df
    .duplicated(
        subset=[
            "condition_id",
            "period",
        ]
    )
    .sum()
)

csv_checks = {
    "CSV 파일 존재": csv_path.is_file(),
    "저장 행 수 일치": (
        len(verified_csv_df)
        == len(trend_df)
    ),
    "열 수 일치": (
        len(verified_csv_df.columns)
        == len(export_columns)
    ),
    "condition_id 결측 없음": (
        verified_csv_df[
            "condition_id"
        ]
        .notna()
        .all()
    ),
    "period 변환 오류 없음": (
        verified_csv_df["period"]
        .notna()
        .all()
    ),
    "ratio 결측 없음": (
        verified_csv_df["ratio"]
        .notna()
        .all()
    ),
    "ratio 0~100 범위": (
        verified_csv_df["ratio"]
        .between(
            0,
            100,
            inclusive="both",
        )
        .all()
    ),
    "CSV 중복 데이터 0개": (
        csv_duplicate_count == 0
    ),
    "gender_code 결측 없음": (
        verified_csv_df[
            "gender_code"
        ]
        .notna()
        .all()
    ),
    "age_code 결측 없음": (
        verified_csv_df[
            "age_code"
        ]
        .notna()
        .all()
    ),
    "전체 성별 all 코드 존재": (
        "all"
        in verified_csv_df[
            "gender_code"
        ].unique()
    ),
    "전체 연령 all 코드 존재": (
        "all"
        in verified_csv_df[
            "age_code"
        ].unique()
    ),
}

all_csv_checks_passed = all(
    bool(result)
    for result in csv_checks.values()
)


# --------------------------------------------------
# 8) 파일 정보 생성
# --------------------------------------------------
csv_file_size = csv_path.stat().st_size

with csv_path.open("rb") as file:
    csv_sha256 = hashlib.sha256(
        file.read()
    ).hexdigest()

CLEAN_CSV_PATH = csv_path


# --------------------------------------------------
# 9) 결과 출력
# --------------------------------------------------
print("=" * 70)
print("STEP 9 분석용 CSV 저장 및 검증 결과")
print("=" * 70)

print(f"저장 파일 : {csv_path}")
print(f"저장 행 수: {len(verified_csv_df):,}행")
print(f"저장 열 수: {len(verified_csv_df.columns)}개")
print(f"파일 크기 : {csv_file_size:,} bytes")
print(f"SHA-256   : {csv_sha256}")

print("\n[CSV 검증]")
for check_name, passed in csv_checks.items():
    status = "정상" if bool(passed) else "확인 필요"
    print(f"[{status}] {check_name}")

print("\n[성별 코드]")
print(
    verified_csv_df[
        "gender_code"
    ].value_counts(dropna=False)
)

print("\n[연령 코드]")
print(
    verified_csv_df[
        "age_code"
    ].value_counts(dropna=False)
    .sort_index()
)

print("-" * 70)

if all_csv_checks_passed:
    print("분석용 CSV 저장 상태: 정상")
else:
    print("분석용 CSV 저장 상태: 확인 필요")

print("=" * 70)

STEP 9 분석용 CSV 저장 및 검증 결과
저장 파일 : C:\Users\user\Desktop\KDT_2026\프로젝트\03_융합_프로젝트\01_DB\data\processed\naver_trend_clean_20261120_192943.csv
저장 행 수: 5,801행
저장 열 수: 19개
파일 크기 : 1,392,990 bytes
SHA-256   : c52ea3e836e752653ac8f751dc9223094bbc1904e6bfcaa571637edb31cebc4b

[CSV 검증]
[정상] CSV 파일 존재
[정상] 저장 행 수 일치
[정상] 열 수 일치
[정상] condition_id 결측 없음
[정상] period 변환 오류 없음
[정상] ratio 결측 없음
[정상] ratio 0~100 범위
[정상] CSV 중복 데이터 0개
[정상] gender_code 결측 없음
[정상] age_code 결측 없음
[정상] 전체 성별 all 코드 존재
[정상] 전체 연령 all 코드 존재

[성별 코드]
gender_code
all    2039
f      1989
m      1773
Name: count, dtype: Int64

[연령 코드]
age_code
1      143
10     481
11     485
2      447
3      515
4      526
5      531
6      544
7      532
8      522
9      505
all    570
Name: count, dtype: Int64
----------------------------------------------------------------------
분석용 CSV 저장 상태: 정상


In [15]:
# STEP 10-A. Python 및 MySQL 분석용 논리 테이블 구성

import json
import hashlib
import pandas as pd

from pathlib import Path


# --------------------------------------------------
# 1) 원본 파일 경로
# --------------------------------------------------
project_root = Path.cwd().resolve()

raw_json_path = (
    project_root
    / "data"
    / "raw"
    / "naver_trend_raw_20261120_192943.json"
)

clean_csv_path = (
    project_root
    / "data"
    / "processed"
    / "naver_trend_clean_20261120_192943.csv"
)

if not raw_json_path.is_file():
    raise FileNotFoundError(
        f"원본 JSON이 없습니다: {raw_json_path}"
    )

if not clean_csv_path.is_file():
    raise FileNotFoundError(
        f"분석용 CSV가 없습니다: {clean_csv_path}"
    )


# --------------------------------------------------
# 2) 원본 JSON과 분석 CSV 불러오기
# --------------------------------------------------
with raw_json_path.open(
    mode="r",
    encoding="utf-8",
) as file:
    raw_data = json.load(file)

raw_metadata = raw_data["metadata"]
raw_results = raw_data["results"]

analysis_df = pd.read_csv(
    clean_csv_path,
    encoding="utf-8-sig",
    dtype={
        "condition_id": "string",
        "normalization_scope_id": "string",
        "gender_code": "string",
        "age_code": "string",
    },
)

for column in [
    "period",
    "start_date",
    "end_date",
]:
    analysis_df[column] = pd.to_datetime(
        analysis_df[column],
        errors="raise",
    )

analysis_df["ratio"] = pd.to_numeric(
    analysis_df["ratio"],
    errors="raise",
)


# --------------------------------------------------
# 3) 수집 ID 생성
# --------------------------------------------------
source_timestamp = (
    raw_json_path
    .stem
    .removeprefix("naver_trend_raw_")
)

collection_id = f"NT_{source_timestamp}"

analysis_df.insert(
    0,
    "collection_id",
    collection_id,
)


# --------------------------------------------------
# 4) 수집 이력 테이블 구성
# --------------------------------------------------
def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            sha256.update(block)

    return sha256.hexdigest()


summary = raw_metadata["collection_summary"]

collection_df = pd.DataFrame(
    [
        {
            "collection_id": collection_id,
            "source_name": raw_metadata["source"],
            "collected_at": summary["finished_at"],
            "saved_at": raw_metadata["saved_at"],
            "start_date": (
                raw_metadata[
                    "collection_period"
                ]["start_date"]
            ),
            "end_date": (
                raw_metadata[
                    "collection_period"
                ]["end_date"]
            ),
            "time_unit": (
                raw_metadata[
                    "collection_period"
                ]["time_unit"]
            ),
            "request_count": summary[
                "collected_condition_count"
            ],
            "success_count": summary[
                "success_count"
            ],
            "data_condition_count": summary[
                "success_with_data_count"
            ],
            "empty_condition_count": summary[
                "success_empty_count"
            ],
            "failure_count": summary[
                "failure_count"
            ],
            "monthly_row_count": len(analysis_df),
            "raw_json_file": raw_json_path.name,
            "clean_csv_file": clean_csv_path.name,
            "raw_json_sha256": calculate_sha256(
                raw_json_path
            ),
            "clean_csv_sha256": calculate_sha256(
                clean_csv_path
            ),
        }
    ]
)

for column in [
    "collected_at",
    "saved_at",
]:
    collection_df[column] = pd.to_datetime(
        collection_df[column],
        errors="raise",
    )

for column in [
    "start_date",
    "end_date",
]:
    collection_df[column] = pd.to_datetime(
        collection_df[column],
        errors="raise",
    )


# --------------------------------------------------
# 5) 요청 조건 테이블 구성
# --------------------------------------------------
condition_rows = []

for result in raw_results:
    condition = result["condition"]

    condition_rows.append(
        {
            "collection_id": collection_id,
            "condition_id": result[
                "condition_id"
            ],
            "normalization_scope_id": condition[
                "normalization_scope_id"
            ],
            "category": condition["category"],
            "keyword": condition["keyword"],
            "gender_code": (
                condition.get("gender_code")
                or "all"
            ),
            "gender_label": condition[
                "gender_label"
            ],
            "age_code": (
                condition.get("age_code")
                or "all"
            ),
            "age_label": condition[
                "age_label"
            ],
            "start_date": condition[
                "start_date"
            ],
            "end_date": condition["end_date"],
            "time_unit": condition[
                "time_unit"
            ],
            "request_status": result[
                "request_status"
            ],
            "http_status": result[
                "http_status"
            ],
            "attempt_count": result[
                "attempt_count"
            ],
            "data_point_count": result[
                "data_point_count"
            ],
            "source_file": raw_json_path.name,
        }
    )

condition_df = pd.DataFrame(
    condition_rows
)

condition_df["start_date"] = pd.to_datetime(
    condition_df["start_date"],
    errors="raise",
)

condition_df["end_date"] = pd.to_datetime(
    condition_df["end_date"],
    errors="raise",
)

condition_df["http_status"] = pd.to_numeric(
    condition_df["http_status"],
    errors="coerce",
).astype("Int64")

condition_df["attempt_count"] = pd.to_numeric(
    condition_df["attempt_count"],
    errors="raise",
).astype("Int64")

condition_df["data_point_count"] = pd.to_numeric(
    condition_df["data_point_count"],
    errors="raise",
).astype("Int64")


# --------------------------------------------------
# 6) 월별 지수 테이블 구성
# --------------------------------------------------
monthly_df = analysis_df[
    [
        "collection_id",
        "condition_id",
        "period",
        "ratio",
    ]
].copy()

monthly_df["ratio"] = monthly_df[
    "ratio"
].astype("float64")


# --------------------------------------------------
# 7) 관계와 무결성 검증
# --------------------------------------------------
condition_primary_key = [
    "collection_id",
    "condition_id",
]

monthly_primary_key = [
    "collection_id",
    "condition_id",
    "period",
]

condition_duplicate_count = (
    condition_df
    .duplicated(
        subset=condition_primary_key
    )
    .sum()
)

monthly_duplicate_count = (
    monthly_df
    .duplicated(
        subset=monthly_primary_key
    )
    .sum()
)

condition_keys = set(
    zip(
        condition_df["collection_id"],
        condition_df["condition_id"],
    )
)

monthly_condition_keys = set(
    zip(
        monthly_df["collection_id"],
        monthly_df["condition_id"],
    )
)

empty_condition_ids = set(
    condition_df.loc[
        condition_df["request_status"].eq(
            "success_empty"
        ),
        "condition_id",
    ]
)

monthly_condition_ids = set(
    monthly_df["condition_id"]
)

data_condition_ids = set(
    condition_df.loc[
        condition_df["request_status"].eq(
            "success_with_data"
        ),
        "condition_id",
    ]
)

structure_checks = {
    "수집 이력 1행": (
        len(collection_df) == 1
    ),
    "요청 조건 396행": (
        len(condition_df) == 396
    ),
    "월별 데이터 5,801행": (
        len(monthly_df) == 5801
    ),
    "요청 조건 기본키 중복 없음": (
        condition_duplicate_count == 0
    ),
    "월별 데이터 기본키 중복 없음": (
        monthly_duplicate_count == 0
    ),
    "모든 월별 데이터에 조건 존재": (
        monthly_condition_keys
        .issubset(condition_keys)
    ),
    "데이터 조건과 월별 조건 일치": (
        monthly_condition_ids
        == data_condition_ids
    ),
    "빈 응답 조건에 월별 행 없음": (
        len(
            empty_condition_ids
            & monthly_condition_ids
        )
        == 0
    ),
    "정규화 범위 ID 중복 없음": (
        condition_df[
            "normalization_scope_id"
        ].nunique()
        == 396
    ),
    "수집 실패 0개": (
        collection_df.loc[
            0,
            "failure_count",
        ]
        == 0
    ),
}

all_structure_checks_passed = all(
    bool(result)
    for result in structure_checks.values()
)


# --------------------------------------------------
# 8) 결과 출력
# --------------------------------------------------
print("=" * 70)
print("STEP 10-A 분석 데이터 구조 검증 결과")
print("=" * 70)

print(
    f"collection_df : "
    f"{len(collection_df):,}행 × "
    f"{len(collection_df.columns)}열"
)

print(
    f"condition_df  : "
    f"{len(condition_df):,}행 × "
    f"{len(condition_df.columns)}열"
)

print(
    f"monthly_df    : "
    f"{len(monthly_df):,}행 × "
    f"{len(monthly_df.columns)}열"
)

print("\n[구조 검증]")
for check_name, passed in structure_checks.items():
    status = "정상" if bool(passed) else "확인 필요"
    print(f"[{status}] {check_name}")

print("\n[요청 상태]")
print(
    condition_df[
        "request_status"
    ].value_counts()
)

print("\n[테이블 관계]")
print(
    "collection_df 1행 "
    "→ condition_df 396행 "
    "→ monthly_df 5,801행"
)

print("-" * 70)

if all_structure_checks_passed:
    print("분석 데이터 구조: 정상")
else:
    print("분석 데이터 구조: 확인 필요")

print("=" * 70)

STEP 10-A 분석 데이터 구조 검증 결과
collection_df : 1행 × 17열
condition_df  : 396행 × 17열
monthly_df    : 5,801행 × 4열

[구조 검증]
[정상] 수집 이력 1행
[정상] 요청 조건 396행
[정상] 월별 데이터 5,801행
[정상] 요청 조건 기본키 중복 없음
[정상] 월별 데이터 기본키 중복 없음
[정상] 모든 월별 데이터에 조건 존재
[정상] 데이터 조건과 월별 조건 일치
[정상] 빈 응답 조건에 월별 행 없음
[정상] 정규화 범위 ID 중복 없음
[정상] 수집 실패 0개

[요청 상태]
request_status
success_with_data    342
success_empty         54
Name: count, dtype: int64

[테이블 관계]
collection_df 1행 → condition_df 396행 → monthly_df 5,801행
----------------------------------------------------------------------
분석 데이터 구조: 정상
